In [ ]:
# 1) Imports
import pandas as pd
import plotly.graph_objects as go
import unicodedata



import re

In [ ]:
# 2) Definição de variáveis / parâmetros
CSV_MAMOGRAFIA = "/home/anna/tcc/Tratamento/Mamografia/tabela_mamografia.csv"

# Filtros (cada linha do CSV = um município)
MUNICIPIO        = None
CODIGO_MUNICIPIO = None

# Intervalo de anos
ANOS_RANGE = (2013, 2025)
ANOS_PASSO = 1

# Faixas etárias (barras) — deve bater com o sufixo das colunas do CSV
FAIXAS_ETARIAS = [
    "0-9", "10-19", "20-29", "30-39",
    "40-49", "50-59", "60-69", "70+",
]

# Cores e título
COR_BARRAS = "#2D9CDB"
COR_PONTOS = "#D7263D"
TITULO     = "Mamografia por faixa etária (barras) + total por ano (pontos)"

In [ ]:
# 3) Ajustes / tratamento dos dados (leitura, filtro, somatórios)
def _norm_txt(x):
    if x is None:
        return ""
    x = str(x).strip().lower()
    x = unicodedata.normalize("NFKD", x)
    return "".join(ch for ch in x if not unicodedata.combining(ch))


def _norm_series(s):
    return s.astype(str).map(_norm_txt)


def _fmt_int(n):
    try:
        return f"{int(n):,}".replace(",", ".")
    except Exception:
        return str(n)


def _col_ano_faixa(ano, faixa):
    return f"{int(ano)}_{faixa}"


def _col_total_candidates(ano):
    a = int(ano)
    return [f"total_{a}", f"Total_{a}", f"TOTAL_{a}"]


def _pick_existing_col(frame, candidates):
    for c in candidates:
        if c in frame.columns:
            return c
    return None


def _sum_row_wise(frame, cols):
    """Replica df[col1] + df[col2] + ... (por linha)."""
    if not cols:
        return pd.Series([0] * len(frame), index=frame.index, dtype="int64")
    return (
        frame[cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
        .sum(axis=1)
        .round()
        .astype(int)
    )


df = df_mamografia.copy() if "df_mamografia" in globals() else pd.read_csv(CSV_MAMOGRAFIA)
f = df.copy()

# Padroniza nomes esperados (se vierem com variação)
col_muni = "Municípios" if "Municípios" in f.columns else ("Municipios" if "Municipios" in f.columns else None)
col_cod  = "Código" if "Código" in f.columns else ("Codigo" if "Codigo" in f.columns else None)
if col_muni is None or col_cod is None:
    raise ValueError("CSV_MAMOGRAFIA precisa ter colunas 'Municípios' e 'Código' (ou variações sem acento).")

if MUNICIPIO is not None:
    f = f[_norm_series(f[col_muni]) == _norm_txt(MUNICIPIO)]
if CODIGO_MUNICIPIO is not None:
    f = f[pd.to_numeric(f[col_cod], errors="coerce").astype("Int64") == int(CODIGO_MUNICIPIO)]

if f.empty:
    raise ValueError("Filtro não encontrou nenhuma linha.")

print(f"Linhas selecionadas: {len(f)}")

ano_ini, ano_fim = map(int, ANOS_RANGE)
anos = list(range(ano_ini, ano_fim + 1))

# Valida faixas com base no cabeçalho
faixas_ok = []
for fx in FAIXAS_ETARIAS:
    # se existir pelo menos uma coluna (em qualquer ano) para a faixa, mantém
    pad = re.compile(rf"^[0-9]{{4}}_{re.escape(fx)}$")
    if any(pad.match(c) for c in f.columns):
        faixas_ok.append(fx)
if not faixas_ok:
    raise ValueError("Nenhuma faixa etária de FAIXAS_ETARIAS foi encontrada no CSV. Verifique nomes como '70+' vs '70 ou mais'.")
FAIXAS_ETARIAS = faixas_ok
print("Faixas detectadas:", FAIXAS_ETARIAS)

# Barras: soma por faixa etária no intervalo
por_faixa = {}
for fx in FAIXAS_ETARIAS:
    cols = [_col_ano_faixa(a, fx) for a in anos if _col_ano_faixa(a, fx) in f.columns]
    por_faixa[fx] = int(_sum_row_wise(f, cols).sum())

# Pontos: total por ano (prioriza colunas total_ano; senão soma faixas do ano)
por_ano = {}
for ano in anos:
    col_total = _pick_existing_col(f, _col_total_candidates(ano))
    if col_total is not None:
        por_ano[ano] = int(pd.to_numeric(f[col_total], errors="coerce").fillna(0).sum())
    else:
        cols = [_col_ano_faixa(ano, fx) for fx in FAIXAS_ETARIAS if _col_ano_faixa(ano, fx) in f.columns]
        por_ano[ano] = int(_sum_row_wise(f, cols).sum())

por_ano_s = pd.Series(por_ano).sort_index()

print("Somatório por faixa etária:")
for fx, v in por_faixa.items():
    print(f"  {fx}: {_fmt_int(v)}")
print(f"\nTotal geral (faixas): {_fmt_int(sum(por_faixa.values()))}")

In [ ]:
# 4) Design e layout do gráfico (Plotly) - versão simplificada
import plotly.io as pio

# Força um renderer compatível com VS Code/Jupyter (evita "não aparece" em alguns ambientes)
if "vscode" in pio.renderers:
    pio.renderers.default = "vscode"
elif "plotly_mimetype" in pio.renderers:
    pio.renderers.default = "plotly_mimetype"
elif "notebook_connected" in pio.renderers:
    pio.renderers.default = "notebook_connected"

fig = go.Figure()

# Barras horizontais simples (sem pontos adicionais)
valores_barras = [por_faixa[fx] for fx in FAIXAS_ETARIAS]
tooltips_barras = [
    f"Faixa etária: {fx}\nQuantidade: {_fmt_int(por_faixa[fx])}"
    for fx in FAIXAS_ETARIAS
]

fig.add_trace(go.Bar(
    name=f"Total por faixa etária ({ano_ini}–{ano_fim})",
    x=valores_barras,
    y=FAIXAS_ETARIAS,
    orientation="h",
    marker=dict(color=COR_BARRAS, opacity=0.9),
    hovertext=tooltips_barras,
    hoverinfo="text",
))

# Layout mais compacto e simples
x_max_data = max(valores_barras) if valores_barras else 0
if x_max_data <= 0:
    raise ValueError("Não há valores positivos para plotar (verifique filtros e intervalo de anos).")

fig.update_layout(
    title=dict(text=TITULO, x=0.5),
    height=480,
    width=800,
    plot_bgcolor="#ffffff",
    paper_bgcolor="white",
    margin=dict(l=80, r=40, t=60, b=60),
    hovermode="closest",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    xaxis=dict(title="Quantidade", range=[0, x_max_data * 1.1]),
    yaxis=dict(title="Faixa etária", autorange="reversed"),
)

fig.show()



In [ ]:
# 5) Novo gráfico de mamografias no estilo do gráfico de mortalidade
import matplotlib.pyplot as plt

# Reaproveita os dados já preparados nas células anteriores:
# por_faixa, por_ano_s, ano_ini, ano_fim, FAIXAS_ETARIAS, ANOS_PASSO, COR_BARRAS, COR_PONTOS
TAMANHO_FIG = (24, 10)
X_TICKS = list(range(1, 11)) + [100, 200, 300, 500, 700, 900, 1000, 1100, 1300, 1500]
X_MIN = 0
X_SCALE = "symlog"
X_LINTHRESH = 10
X_LINSCALE = 1.0
COR_GRADE = globals().get("COR_GRADE", "#B0B0B0")
MOSTRAR_ROTULOS_BARRAS = True
MOSTRAR_ROTULOS_PONTOS = True

def ticks_inclusive(inicio, fim, passo):
    ticks = list(range(int(inicio), int(fim) + 1, max(1, int(passo))))
    if ticks and ticks[-1] != int(fim):
        ticks.append(int(fim))
    return ticks

def _fmt_int(n):
    try:
        return f"{int(n):,}".replace(",", ".")
    except Exception:
        return str(n)

por_faixa_s = pd.Series(por_faixa).reindex(FAIXAS_ETARIAS).fillna(0).astype(int)

x_max = int(max(int(por_faixa_s.max()), int(por_ano_s.max())))
x_max = max(x_max, max(X_TICKS) if X_TICKS else 1)
x_min_for_scale = 0.0001 if X_SCALE == "symlog" else X_MIN

plt.style.use("seaborn-v0_8-whitegrid")
fig, ax1 = plt.subplots(figsize=TAMANHO_FIG)

if X_SCALE == "symlog":
    ax1.set_xscale("symlog", linthresh=X_LINTHRESH, linscale=X_LINSCALE, base=10)
elif X_SCALE != "linear":
    raise ValueError("X_SCALE precisa ser 'linear' ou 'symlog'.")

barras = ax1.barh(
    FAIXAS_ETARIAS,
    por_faixa_s.values.tolist(),
    color=COR_BARRAS,
    alpha=0.85,
    label=f"Total por faixa etária (soma {ano_ini}–{ano_fim})",
    zorder=2,
)
ax1.set_xlabel("Número de mamografias")
ax1.set_ylabel("Faixa etária")
ax1.set_xlim(x_min_for_scale, x_max * 1.05)

xticks = [t for t in X_TICKS if X_MIN <= t <= x_max]
ax1.set_xticks(xticks)
ax1.set_xticklabels([str(t) for t in xticks])
ax1.tick_params(axis="x", labelsize=10, pad=10)
ax1.grid(axis="x", linestyle="--", linewidth=0.8, alpha=0.35, color=COR_GRADE)
ax1.grid(axis="y", visible=False)
ax1.set_axisbelow(True)

ax2 = ax1.twinx()
valores_x = por_ano_s.values.tolist()
valores_y = por_ano_s.index.tolist()
scatter = ax2.scatter(
    valores_x,
    valores_y,
    color=COR_PONTOS,
    s=70,
    alpha=0.9,
    edgecolors="white",
    linewidths=0.8,
    label=f"Total por ano ({ano_ini}–{ano_fim})",
    zorder=3,
)
ax2.set_ylabel("Ano")
ax2.set_ylim(ano_ini - 0.5, ano_fim + 0.5)
ax2.set_yticks(ticks_inclusive(ano_ini, ano_fim, ANOS_PASSO))
ax2.tick_params(axis="y", labelsize=10)

if MOSTRAR_ROTULOS_BARRAS:
    for rect in barras.patches:
        val = int(round(rect.get_width()))
        if val <= 0:
            continue
        ax1.annotate(
            _fmt_int(val),
            (val, rect.get_y() + rect.get_height() / 2),
            textcoords="offset points",
            xytext=(8, 0),
            va="center",
            ha="left",
            fontsize=10,
        )

if MOSTRAR_ROTULOS_PONTOS:
    for ano, val in por_ano_s.items():
        val = int(val)
        if val <= 0:
            continue
        ax2.annotate(
            _fmt_int(val),
            (val, int(ano)),
            textcoords="offset points",
            xytext=(8, 0),
            va="center",
            ha="left",
            fontsize=10,
            color=COR_PONTOS,
        )

ax1.legend(loc="upper left", frameon=True)
ax2.legend(loc="upper right", frameon=True)
plt.title("Mamografia por faixa etária (barras) + total por ano (pontos)", fontsize=13, pad=14)
fig.subplots_adjust(bottom=0.18)
plt.tight_layout()
plt.show()